# Build Classification Model

In [1]:
import pandas as pd

cuisines_df = pd.read_csv("../data/cleaned_cuisines.csv")
cuisines_df.head()

,Unnamed: 0,cuisine,almond,angelica,anise,anise_seed,apple,apple_brandy,apricot,armagnac,...,whiskey,white_bread,white_wine,whole_grain_wheat_flour,wine,wood,yam,yeast,yogurt,zucchini
0,0,indian,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,1,indian,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,2,indian,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,3,indian,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,4,indian,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0


In [2]:
cuisines_label_df = cuisines_df["cuisine"]
cuisines_label_df.head()

0    indian
1    indian
2    indian
3    indian
4    indian
Name: cuisine, dtype: str

In [3]:
cuisines_feature_df = cuisines_df.drop(["Unnamed: 0", "cuisine"], axis=1)
cuisines_feature_df.head()

,almond,angelica,anise,anise_seed,apple,apple_brandy,apricot,armagnac,artemisia,artichoke,...,whiskey,white_bread,white_wine,whole_grain_wheat_flour,wine,wood,yam,yeast,yogurt,zucchini
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0


<h3 style="font-weight:600;color:gold;">1 | Split the Data</h3>

In [4]:
# 1. Import the needed libraries
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    confusion_matrix,
    classification_report,
    precision_recall_curve,
)
import numpy as np

# 2. Split training and test data
X_train, X_test, y_train, y_test = train_test_split(cuisines_feature_df, cuisines_label_df, test_size=0.3)

<h3 style="font-weight:600;color:gold;">2 | Linear SVC Classifier</h3>

Support-Vector Clustering (SVC) involves choosing a **kernel** to handle the clustering of the labels. The arguments include:

- **C** : regularization; it regulates the influence of parameters.
- **kernel** : here it is *linear*, to ensure we leverage linear SVC.
- **probability** : set to True to gather probability estimates.
- **random_state** : set to 0 , to shuffle the data to get probabilities.

<h4 style="font-weight:600;color:gold;">2.1 | Apply a Linear SVC</h4>

1. Start with a Linear SVC:

In [5]:
C = 10
# Create different classifiers
classifiers = {
    "Linear SVC": SVC(C=C, kernel="linear", probability=True, random_state=0),
}

2. Train the model using the **Linear SVC** and print out a report:

In [6]:
def classifiers_report():
    n_classifiers = len(classifiers)

    for index, (name, classifier) in enumerate(classifiers.items()):
        classifier.fit(X_train, y_train)
        y_pred = classifier.predict(X_test)
        accuracy = accuracy_score(y_test, y_pred)

        print(f"Accuracy (train) for {name}: {accuracy*100:0.1f}%")
        print(classification_report(y_test, y_pred))


classifiers_report()

Accuracy (train) for Linear SVC: 78.6%
              precision    recall  f1-score   support

     chinese       0.73      0.75      0.74       267
      indian       0.88      0.88      0.88       232
    japanese       0.74      0.73      0.73       227
      korean       0.80      0.77      0.78       224
        thai       0.79      0.80      0.80       249

    accuracy                           0.79      1199
   macro avg       0.79      0.79      0.79      1199
weighted avg       0.79      0.79      0.79      1199



The result is pretty good!

<h3 style="font-weight:600;color:gold;">3 | K-Neighbors Classifier</h3>

**K-Neighbors** can be used for both supervised and unsupervised learning. Start by defining a predifined number of points and then gather data around these points.

<h4 style="font-weight:600;color:gold;">3.1 | Apply the K-Neighbors Classifier</h4>

In [7]:
classifiers["KNN Classifer"] = KNeighborsClassifier(C)

In [8]:
classifiers_report()

Accuracy (train) for Linear SVC: 78.6%
              precision    recall  f1-score   support

     chinese       0.73      0.75      0.74       267
      indian       0.88      0.88      0.88       232
    japanese       0.74      0.73      0.73       227
      korean       0.80      0.77      0.78       224
        thai       0.79      0.80      0.80       249

    accuracy                           0.79      1199
   macro avg       0.79      0.79      0.79      1199
weighted avg       0.79      0.79      0.79      1199

Accuracy (train) for KNN Classifer: 73.0%
              precision    recall  f1-score   support

     chinese       0.73      0.66      0.69       267
      indian       0.87      0.81      0.84       232
    japanese       0.60      0.82      0.69       227
      korean       0.89      0.55      0.68       224
        thai       0.69      0.80      0.74       249

    accuracy                           0.73      1199
   macro avg       0.76      0.73      0.73      1

The result is a little worse: accuracy is now 75.5% instead of 78.0%

<h3 style="font-weight:600;color:gold;">4 | Support Vector Classifier</h3>

Maps training examples to points in space, to maximize the distance between two categories.

<h4 style="font-weight:600;color:gold;">4.1 | Apply a Support Vector Classifier</h4>

In [9]:
classifiers["SVC"] = SVC()
classifiers_report()

Accuracy (train) for Linear SVC: 78.6%
              precision    recall  f1-score   support

     chinese       0.73      0.75      0.74       267
      indian       0.88      0.88      0.88       232
    japanese       0.74      0.73      0.73       227
      korean       0.80      0.77      0.78       224
        thai       0.79      0.80      0.80       249

    accuracy                           0.79      1199
   macro avg       0.79      0.79      0.79      1199
weighted avg       0.79      0.79      0.79      1199

Accuracy (train) for KNN Classifer: 73.0%
              precision    recall  f1-score   support

     chinese       0.73      0.66      0.69       267
      indian       0.87      0.81      0.84       232
    japanese       0.60      0.82      0.69       227
      korean       0.89      0.55      0.68       224
        thai       0.69      0.80      0.74       249

    accuracy                           0.73      1199
   macro avg       0.76      0.73      0.73      1

<h3 style="font-weight:600;color:gold;">5 | Ensemble Classifiers</h3>

Let's try some Ensemble Classifiers such as **Random Forest** and **AdaBoost**.

In [11]:
classifiers["RFST"] = RandomForestClassifier(n_estimators=100)
classifiers["ADA"] = AdaBoostClassifier(n_estimators=100)

classifiers_report()

Accuracy (train) for Linear SVC: 78.6%
              precision    recall  f1-score   support

     chinese       0.73      0.75      0.74       267
      indian       0.88      0.88      0.88       232
    japanese       0.74      0.73      0.73       227
      korean       0.80      0.77      0.78       224
        thai       0.79      0.80      0.80       249

    accuracy                           0.79      1199
   macro avg       0.79      0.79      0.79      1199
weighted avg       0.79      0.79      0.79      1199

Accuracy (train) for KNN Classifer: 73.0%
              precision    recall  f1-score   support

     chinese       0.73      0.66      0.69       267
      indian       0.87      0.81      0.84       232
    japanese       0.60      0.82      0.69       227
      korean       0.89      0.55      0.68       224
        thai       0.69      0.80      0.74       249

    accuracy                           0.73      1199
   macro avg       0.76      0.73      0.73      1

The result is very good especially for <span style="font-weight:600;color:red;">Random Forest</span> with an accuracy of 83.2%.